# Representative doses

Boxplots and summary statistics of DAP/CAK/exposure time per procedure and lab.

Set `ANALYSIS` to one of the analyses in `config.toml` (`pci`, `elfys`, `radiology`, `dsa`, `pediatric`) and `YEAR` to the year to analyse (`None` reads every year under the data folders).

In [ ]:
%load_ext autoreload
%autoreload 2

import xa_dose_analysis as xa

xa.setup_logging("INFO")  # use "WARNING" for less output
cfg = xa.load_config()

In [ ]:
ANALYSIS = "pci"
YEAR = 2025

In [ ]:
# Read, clean and merge the IDS7 and DoseTrack exports:
ds = xa.load_dataset(cfg, year=YEAR)
ds.quality.to_frame()

In [ ]:
# Filter to the rooms of the analysis and map the procedure descriptions:
data = xa.select_analysis(ds, cfg, ANALYSIS)
data.head()

In [ ]:
# Which descriptions were not recognized by the mapping?
ds.quality.unmapped_descriptions[ANALYSIS].head(20)

In [ ]:
# Boxplot per configured procedure (save=True writes PNGs to the Figures folder):
xa.plot_representative_doses(data, cfg.analysis(ANALYSIS), save=False)

In [ ]:
# Summary statistics (median, bootstrap 95% CI, IQR, range) per procedure and room:
stats = xa.summary_by_procedure(data, seed=1)
xa.print_summary(stats)
stats

In [ ]:
# Compare against the diagnostic reference levels configured in config.toml:
xa.compare_with_drl(stats, cfg.analysis(ANALYSIS))

In [ ]:
# Export the statistics to Excel:
xa.export_summary({f"{ANALYSIS} {YEAR}": stats}, cfg.reports_dir / f"{ANALYSIS}_{YEAR}.xlsx")